# Set up the Fabric policy - new workspaces

Use this notebook to **create the three workspaces** on a Fabric capacity and set up the policy. It checks the
capacity, creates the workspaces (reusing any that already exist), creates a **policy set** in the **policy-store**
workspace, adds three **rules** to it, and **turns it on** for the capacity. The steps are explained just before the
cell that runs them.

| Workspace | Setting | Purpose |
| --- | --- | --- |
| **Policy store** | `POLICY_STORE_WORKSPACE` | Stores the policy set. It isn't restricted. |
| **Data Engineering** | `DATA_ENGINEERING_WORKSPACE` | Can create only Lakehouse, Data pipeline and Notebook. |
| **Data Engineering + AI** | `DATA_ENGINEERING_AI_WORKSPACE` | Can create the same items, plus Data agent. |

| Rule | Applies to | Allows |
| --- | --- | --- |
| 1 | Data Engineering and Data Engineering + AI workspaces | Lakehouse, Data pipeline, Notebook |
| 2 | Data Engineering + AI workspace | Data agent |
| 3 | Every other workspace on the capacity | Everything |

So, after up to 15 minutes, the **Data Engineering** workspace can't create a Data agent, the **Data Engineering +
AI** workspace can, and every other workspace on the capacity works as before.

### Before you start

* Import this notebook **and** `policy_setup_functions.ipynb` into the **same workspace**.
* To create workspaces, you need the **Capacity Contributor** or **Capacity Admin** role on the capacity, and your
  Fabric administrator must allow you to create workspaces (the **Create workspaces** tenant setting).
* Your Fabric administrator has turned on the **Policies** tenant setting for you.
* You're the Admin of the workspaces the notebook creates. If a workspace **already exists**, it's reused, and you
  need the **Contributor**, **Member** or **Admin** role on the **policy-store** workspace (the policy set is created
  in it). For the **Data Engineering** and **Data Engineering + AI** workspaces, the **Viewer** role is enough.
* To turn the policy set on, you need the **Capacity Admin** role. Without it, steps 1-4 still run, and step 5
  prints a message for your Capacity Admin.

### How to run

Type the capacity ID and workspace names in the **Settings** cell, and then select **Run all**. To find the
capacity ID, go to **OneLake catalog** > **Govern** > **Capacities** (you need permission to view capacity
information). Not sure of the capacity ID? Run it anyway: if the ID isn't right, step 1 lists the capacities you
can use, with their IDs. Each step says what it did. If something goes wrong, the notebook stops and tells you
what to do. It's safe to run again.

## Settings - type the capacity ID and workspace names here

In [ ]:
# To find the capacity ID: OneLake catalog > Govern > Capacities (you need permission to view capacity information)
CAPACITY_ID = "00000000-0000-0000-0000-000000000000"   # the Fabric capacity to create the workspaces on

POLICY_STORE_WORKSPACE = "Policy-Admin-Store"                 # Policy store: stores the policy set
DATA_ENGINEERING_WORKSPACE = "Policy-DataEngineering"         # Data Engineering: Lakehouse, Data pipeline, Notebook
DATA_ENGINEERING_AI_WORKSPACE = "Policy-DataEngineering-AI"   # Data Engineering + AI: can also create a Data agent

POLICY_SET_NAME = "Fabric_ItemCreation_Governance"   # no spaces

## Load the setup functions

Loads the notebook **policy_setup_functions**, which has the code for every step below. It must be in the **same
workspace** as this notebook. If this cell says the notebook can't be found, import `policy_setup_functions.ipynb` into this
workspace (keep its name), and run again.

In [ ]:
%run policy_setup_functions

## Set up the policy - steps 1 to 5

The cell below runs all five steps, in order. Each step prints what it did. If a step fails, the notebook stops
there and tells you what to do.

| Step | What it does | Function |
| --- | --- | --- |
| 1. Check the capacity | Checks that the capacity exists, is running, and that you can create workspaces on it (**Capacity Contributor** or **Capacity Admin**). If the ID is wrong, it lists the capacities you can use. It also tells you whether you're a Capacity Admin. | `check_capacity` |
| 2. Create the workspaces | Creates the three workspaces on the capacity; you're the Admin of the ones it creates. Workspaces that already exist are reused and **never moved**.     Before creating anything, it stops if the **Data Engineering** or **Data Engineering + AI** workspace already exists on another capacity, or if the **policy-store** workspace already exists and you're only a **Viewer** there. | `create_workspaces` |
| 3. Create the policy set | Creates the policy set in the **policy-store** workspace. It stays **off** until step 5. If it already exists, it's reused. | `create_policy_set` |
| 4. Add the rules | Sets the three rules from the table at the top. Running it again replaces them, so they're never duplicated. | `add_rules` |
| 5. Turn on the policy set | Turns it on for the capacity. Needs the **Capacity Admin** role; without it, the policy set and rules stay saved (but off), and you get a message to send to your Capacity Admin. | `turn_on_policy_set` |

The functions are in `policy_setup_functions`, and the rules are in its **Rules** section.

In [ ]:
capacity = check_capacity(CAPACITY_ID)                                         # Step 1
policy_store_ws, data_eng_ws, data_eng_ai_ws = create_workspaces(capacity,     # Step 2
    POLICY_STORE_WORKSPACE, DATA_ENGINEERING_WORKSPACE, DATA_ENGINEERING_AI_WORKSPACE)
policy_set = create_policy_set(policy_store_ws, POLICY_SET_NAME)               # Step 3
add_rules(policy_store_ws, policy_set, data_eng_ws, data_eng_ai_ws)            # Step 4
turn_on_policy_set(policy_store_ws, policy_set, capacity)                      # Step 5

## Try it

After up to 15 minutes:

| Where | Create a... | Result |
| --- | --- | --- |
| Data Engineering | Data agent | Blocked |
| Data Engineering + AI | Data agent | Allowed |
| Data Engineering or Data Engineering + AI | Lakehouse, Data pipeline or Notebook | Allowed |
| Data Engineering or Data Engineering + AI | Anything else, such as a Warehouse | Blocked |
| Policy store, or any other workspace | Anything | Allowed |